# Face Mask Detection with CNN

## Overview
Advance deep learning and data engineering expertise through projects that apply neural networks and data processing techniques to real-world problems. This includes developing data aggregation and visualization modules for healthcare analytics, building predictive models for banking retention, applying computer vision for mask detection, leveraging hybrid models for sentiment analysis, and using autoencoders for medical image denoising. These projects emphasize model design, data handling, and evaluation to strengthen technical proficiency in supervised and unsupervised learning.

## Project Statement
Develop CNN model across diverse domains through a hands-on project that addresses distinct real-world challenges involving image data, emphasizing model design, data preprocessing, and performance evaluation. The projects promote practical understanding of CNNs through industry-relevant problem-solving using neural networks.

## Task: Detect humans wearing face masks

### Requirements:

1. **Load and preprocess the image datasets**
   - Prepare Training and Testing Datasets

2. **Develop and train CNN model**
   - Ensure detailed discussion and well thought out reasoning about Model architecture
   - Include your reasoning for:
	 - Layers
	 - Inputs (up to you to research and decide) and Output Layers
	 - Activation Functions
	 - Hyperparameters
	 - Training and Validation Metrics
	 - Loss Function

3. **Use callbacks and early stopping for efficient optimization**
   - Decide on a reasonable metric to monitor
   - Describe why this is the metric that you should monitor
   - Provide evidence / describe why your early stopping worked

4. **Evaluate and compare model performance**
   - Test Data
   - Visualize the predictions
   - Include the image along with the True & Predicted labels
   - Determine the best-performing model

### Extra Credit (Optional):
- Include Data Augmentation Procedures
- Train a Finalized Model on ALL your data
- Use StreamLit - Drawable Canvas to get live model inference via drawing


In [ ]:
# Imports
# Standard library imports
from pathlib import Path

# Third-party imports
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn, optim
import optuna
from optuna import Trial

from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Set device
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
# Hyperparameters lists of potential values to optimize over
num_classes = 10
learning_rate_l = [1e-4, 1e-3, 1e-2]
initial_filters_l = [16, 32, 64]
n_conv_blocks_l = [2, 3, 4]
batch_size_l = [64, 128, 256]
fc_units_1_l = [16, 32, 64]
fc_units_2_l = [16, 32, 64]
dropout_rate_l = [0.3, 0.4, 0.5]


In [ ]:
# Unzip our data from zip file if data dir does not exist
data_path = Path('data')
if not data_path.is_dir():
    import zipfile
    with zipfile.ZipFile('data.zip', 'r') as zip_ref:
        zip_ref.extractall('.')

# Define dataloaders
train_dir = data_path / 'train'
test_dir = data_path / 'test'
transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
])
train_dataset = datasets.ImageFolder(train_dir, transform=transform)
test_dataset = datasets.ImageFolder(test_dir, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)
num_classes = len(train_dataset.classes)
# print sizes
print(f"Number of training samples: {len(train_dataset)}")
print(f"Number of testing samples: {len(test_dataset)}")
print(f"Number of classes: {num_classes}")

In [ ]:
# Custom loss function using nn.Module as a base class
class CustomLoss(nn.Module):
    def __init__(self, 
                 apply_class_balancing=False,
                 alpha=0.25,
                 gamma=2.0,
                 from_logits=False,
                 label_smoothing=0.0,
                 axis=-1,
                 reduction='sum' # 'none', 'mean', 'sum'
                 ):
        super().__init__()
        self.apply_class_balancing = apply_class_balancing
        self.alpha = alpha
        self.gamma = gamma
        self.from_logits = from_logits
        self.label_smoothing = label_smoothing
        self.axis = axis
        self.reduction = reduction
        self.name = 'binary_focal_crossentropy'

    def forward(self, y_pred, y_true):
        ce_base = nn.BCELoss(reduction=self.reduction,
                                      label_smoothing=self.label_smoothing)
        p_t = ce_base(y_pred, y_true)
        # CE(p_t) = − log(p_t)
        # FL(p_t) = −(1 − p_t)γ * log(p_t)
        # or equivalently:
        # FL(p_t) = (1 - p_t) ** gamma * CE(p_t
        if self.gamma != 0:
            focal_loss = (self.alpha * (1 - p_t) ** self.gamma) * p_t
            return focal_loss
        else:
            # If gamma is 0, focal loss is just cross-entropy loss
            return p_t



In [ ]:
# Create the CNN using sequential layers

def create_cnn(n_conv_blocks: int, dropout_rate: float, initial_filters: int, fc_units_1: int, fc_units_2: int, num_classes: int) -> nn.Sequential:
    '''Create a CNN with configurable architecture.
    
    Args:
        n_conv_blocks: Number of convolutional blocks (1-4)
        dropout_rate: Dropout probability
        fc_units_1: Number of units in the first fully connected layer
        fc_units_2: Number of units in the second fully connected layer
        initial_filters: Number of filters in the first convolutional block
        num_classes: Number of output classes
    
    Returns:
        nn.Sequential model
    '''
    layers = []
    in_channels = 3  # RGB input
    current_size = 32  # Input image size
    
    for block_idx in range(n_conv_blocks):
        out_channels = initial_filters * (2 ** block_idx)
        
        # Conv -> BatchNorm -> ReLU -> Conv -> BatchNorm -> ReLU -> Pool -> Dropout
        layers.append(nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1))
        layers.append(nn.BatchNorm2d(out_channels))
        layers.append(nn.ReLU())
        
        layers.append(nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1))
        layers.append(nn.BatchNorm2d(out_channels))
        layers.append(nn.ReLU())
        
        layers.append(nn.MaxPool2d(2, 2))
        layers.append(nn.Dropout(dropout_rate))
        
        in_channels = out_channels
        current_size //= 2
    
    # Calculate flattened size
    final_channels = initial_filters * (2 ** (n_conv_blocks - 1))
    flattened_size = final_channels * current_size * current_size
    
    # Classifier (3 fully connected layers)
    layers.append(nn.Flatten())
    layers.append(nn.Linear(flattened_size, fc_units_1))
    layers.append(nn.ReLU())
    layers.append(nn.Dropout(dropout_rate))
    layers.append(nn.Linear(fc_units_1, fc_units_2))
    layers.append(nn.ReLU())
    layers.append(nn.Dropout(dropout_rate))
    layers.append(nn.Linear(fc_units_2, num_classes))
    
    return nn.Sequential(*layers)
